# 🔍 Python Search Algorithms — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Searching is asking "where is X?" in a collection. Linear search checks every door down the hall. Binary search works like a phone book — open the middle, ask "too early or too late?", and discard half the book. Jump search leaps in blocks before checking. Exponential search doubles the window until it overshoots. The right strategy depends entirely on whether the data is sorted and how fast you need the answer.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [Visual Model — Search Strategies](#1) |
| 2 | [Complexity at a Glance](#2) |
| 3 | [Decision Map — Which Search When](#3) |
| 4 | [Pattern 1: Linear Search & Sentinel](#4) |
| 5 | [Pattern 2: Jump Search](#5) |
| 6 | [Pattern 3: Exponential Search](#6) |
| 7 | [Pattern 4: Binary Search — Standard Template](#7) |
| 8 | [Pattern 5: Lower Bound (leftmost)](#8) |
| 9 | [Pattern 6: Upper Bound (rightmost)](#9) |
| 10 | [Pattern 7: Search on Answer Space](#10) |
| 11 | [Pattern 8: Rotated Sorted Array](#11) |
| 12 | [Pattern 9: bisect module](#12) |
| 13 | [LC 34 — First and Last Position](#13) |
| 14 | [LC 875 — Koko Eating Bananas (answer space)](#14) |
| 15 | [Full Decision Map](#15) |
| 16 | [Interview Cheat Sheet](#16) |


<a id='1'></a>

## 1. Visual Model — Search Strategies

```
LINEAR SEARCH — check every element
arr = [3, 7, 1, 9, 4]   target=9
  check 3 ✗   check 7 ✗   check 1 ✗   check 9 ✓  found at index 3

BINARY SEARCH — sorted array only, cut in half each step
arr = [1, 3, 4, 7, 9, 12, 15]   target=9
  lo=0  hi=6  mid=3  arr[3]=7 < 9  → lo=4
  lo=4  hi=6  mid=5  arr[5]=12 > 9 → hi=4
  lo=4  hi=4  mid=4  arr[4]=9 = 9  ✓ found!
  3 comparisons for 7 elements — vs 5 for linear

JUMP SEARCH — jump √n steps, then linear within block
arr = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]   target=13   √10≈3
  jump to index 3: arr[3]=7 < 13  jump
  jump to index 6: arr[6]=13 = 13 ✓  found in 2 jumps

EXPONENTIAL SEARCH — double window until overshoot, then binary
arr = [1, 2, 4, 8, 16, 32, 64, 128]   target=32
  i=1: arr[1]=2 < 32  double
  i=2: arr[2]=4 < 32  double
  i=4: arr[4]=16 < 32 double
  i=8: arr[8] out of range — overshoot at i=8
  binary search in [i//2, min(i, n-1)] = [4, 7]
  found at index 5 ✓

BINARY SEARCH TEMPLATES:
  lo=0  hi=n-1          standard: exact target
  lo=0  hi=n-1          lower_bound: leftmost target (<=)
  lo=0  hi=n-1          upper_bound: rightmost target (>=)
  lo=min hi=max         answer space: minimize/maximize feasible value
```


<a id='2'></a>

## 2. Complexity at a Glance

```
ALGORITHM            TIME (avg/worst)   SPACE   SORTED?   NOTES
──────────────────────────────────────────────────────────────────────────
Linear Search        O(n) / O(n)        O(1)    NO        baseline
Sentinel Search      O(n) / O(n)        O(1)    NO        fewer comparisons/iter
Jump Search          O(√n) / O(√n)      O(1)    YES       block size = √n
Exponential Search   O(log n) / O(log n) O(1)   YES       for unbounded arrays
Binary Search        O(log n) / O(log n) O(1)   YES       divide & conquer
Fibonacci Search     O(log n) / O(log n) O(1)   YES       avoids division
bisect_left/right    O(log n) / O(log n) O(1)   YES       Python stdlib
──────────────────────────────────────────────────────────────────────────
```


<a id='3'></a>

## 3. Decision Map — Which Search When

```
SIGNAL IN THE PROBLEM                  USE THIS
─────────────────────────────────────────────────────────────────────────
Unsorted array, find exact value       Linear Search
Sorted array, find exact value         Binary Search (standard)
Sorted array, find leftmost/first      Lower Bound (bisect_left)
Sorted array, find rightmost/last      Upper Bound (bisect_right - 1)
"minimum X such that condition(X)"     Binary Search on answer space
"maximum X such that condition(X)"     Binary Search on answer space
Sorted but rotated                     Binary Search (find-pivot logic)
Unknown array size / infinite          Exponential Search
Sorted, cache-friendly, avoid div/mul  Jump Search
Python: insert position in sorted      bisect.insort
"Is X possible?" feasibility check     Answer space binary search
```


<a id='4'></a>

## 4. 🧩 Pattern 1: Linear Search & Sentinel — LC 744

---

```
PROBLEM:  Find target in unsorted array. No assumptions about order.

APPROACH: Walk element-by-element. Return index when found, -1 if not found.
          Sentinel optimization: append target to end of array — eliminates
          the "i < n" bounds check inside the loop (one fewer comparison per iter).

SLOW MOTION TRACE on arr=[3,7,1,9,4] target=9:
  i=0: 3≠9 continue
  i=1: 7≠9 continue
  i=2: 1≠9 continue
  i=3: 9=9 ✓ return 3

  Sentinel version: arr=[3,7,1,9,4,9]  (9 appended)
  loop until arr[i]==9 — guaranteed to stop without bounds check
  if i == original_length: not found

KEY INSIGHT: Linear search beats binary search when:
  - Array is unsorted (binary search requires sorted)
  - Array is tiny (n < ~20, constant factor dominates)
  - Cache locality matters (sequential access is L1-cache friendly)
  - You need to search a linked list

TIME / SPACE:
  Time:  O(n) — scan up to n elements
  Space: O(1) — no extra memory
```


In [ ]:
from typing import List

def linear_search(arr: List[int], target: int) -> int:
    """
    Linear Search — O(n), works on unsorted arrays.
    Approach: scan left to right, return first matching index.
    Args:
        arr (List[int]): array to search (can be unsorted).
        target (int): value to find.
    Returns:
        int: first index where arr[i] == target, or -1 if not found.
    Time:  O(n) — worst case checks all n elements
    Space: O(1) — no extra memory
    """
    for i, val in enumerate(arr):
        if val == target:
            return i          # found — return position
    return -1                 # not found

def sentinel_search(arr: List[int], target: int) -> int:
    """
    Sentinel Search — same O(n) but eliminates bounds check per iteration.
    Trick: append target to end so the loop always terminates, then check if
    the found index was within original bounds.
    Args:
        arr (List[int]): array to search.
        target (int): value to find.
    Returns:
        int: index if found, -1 if not found.
    Time:  O(n) — same as linear, but ~fewer operations per iteration
    Space: O(1) — modifies arr temporarily (or use a copy)
    """
    n = len(arr)
    arr.append(target)         # sentinel — guarantees loop stops
    i = 0
    while arr[i] != target:    # no bounds check needed — sentinel is always there
        i += 1
    arr.pop()                  # remove sentinel
    return i if i < n else -1  # found in original? or only found sentinel?

# LC 744 — Find Smallest Letter Greater Than Target (linear scan variant)
def nextGreatestLetter(letters: List[str], target: str) -> str:
    """
    LC 744 — Find Smallest Letter Greater Than Target.
    Approach: linear scan — find first letter strictly > target (wrap around).
    Args:
        letters (List[str]): sorted list of lowercase letters (with wrapping).
        target (str): the reference character.
    Returns:
        str: smallest letter in letters strictly greater than target.
    Time:  O(n) — one pass (binary search version also valid but this is clear)
    Space: O(1)
    """
    for letter in letters:
        if letter > target:    # first letter strictly greater — sorted so this is smallest
            return letter
    return letters[0]          # wrap: all letters <= target, return smallest in list

def test_harness_linear(fn):
    tests = [
        ([3, 7, 1, 9, 4], 9, 3),
        ([3, 7, 1, 9, 4], 5, -1),
        ([], 1, -1),
        ([1], 1, 0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]), inputs[1])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed (linear_search)")

def test_harness_next_letter(fn):
    tests = [
        (['c','f','j'], 'a', 'c'),
        (['c','f','j'], 'c', 'f'),
        (['c','f','j'], 'j', 'c'),    # wrap around
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed (nextGreatestLetter)")

test_harness_linear(linear_search)
test_harness_next_letter(nextGreatestLetter)
print("linear_search, sentinel_search, nextGreatestLetter defined.")

# Simplicity and clarity is Gold


<a id='5'></a>

## 5. 🧩 Pattern 2: Jump Search — O(√n) on sorted arrays

---

```
PROBLEM:  Search a sorted array faster than linear but without binary search overhead.
          Useful when backward stepping is costly (e.g., magnetic tape, streaming).

APPROACH: Jump forward by √n steps until overshoot or end.
          Then linear search backward within the block where target could be.

SLOW MOTION TRACE on arr=[0,1,2,3,4,5,6,7,8,9,10,11] target=7  step=√12≈3:
  jump to index 3:  arr[3]=3 < 7  continue jumping
  jump to index 6:  arr[6]=6 < 7  continue jumping
  jump to index 9:  arr[9]=9 > 7  overshoot — target is in [6..9)
  linear search backward from 9:
    arr[9]=9≠7  arr[8]=8≠7  arr[7]=7=7 ✓  found at index 7

KEY INSIGHT: √n jumps + up to √n linear steps = O(√n) total.
             Optimal block size is exactly √n.

TIME / SPACE:
  Time:  O(√n) — ≤ √n jump comparisons + ≤ √n linear comparisons
  Space: O(1)  — no extra memory
```


In [ ]:
from typing import List
import math

def jump_search(arr: List[int], target: int) -> int:
    """
    Jump Search — O(√n) for sorted arrays.
    Approach: jump by √n blocks until overshoot, then linear scan within block.
    Args:
        arr (List[int]): sorted list of integers.
        target (int): value to find.
    Returns:
        int: index of target, or -1 if not found.
    Time:  O(√n) — at most √n jumps + √n linear steps
    Space: O(1)  — constant extra memory
    """
    n = len(arr)
    if n == 0:
        return -1

    step = int(math.sqrt(n))    # optimal block size for O(√n)
    prev = 0

    # Phase 1: jump forward until arr[min(step,n)-1] >= target or end
    while step < n and arr[step] < target:
        prev = step              # remember start of current block
        step += int(math.sqrt(n))  # jump to next block

    # Phase 2: linear search backward within [prev .. min(step, n))
    for i in range(prev, min(step, n)):
        if arr[i] == target:
            return i             # found
        if arr[i] > target:
            return -1            # overshot — target not in array

    return -1

# Slow motion on arr=[0..11], target=7, step=3:
# prev=0, step=3: arr[3]=3<7 → prev=3, step=6
# prev=3, step=6: arr[6]=6<7 → prev=6, step=9
# prev=6, step=9: arr[9]=9>=7 → stop jumping
# linear i=6: arr[6]=6≠7  i=7: arr[7]=7=7 ✓ return 7

def test_harness(fn):
    arr = list(range(12))     # [0,1,2,...,11]
    tests = [
        (arr, 7, 7),
        (arr, 0, 0),
        (arr, 11, 11),
        (arr, 5, -1 if 5 not in arr else 5),   # 5 is in arr
        (arr, 99, -1),
        ([], 1, -1),
    ]
    # fix test 4
    tests[3] = (arr, 5, 5)
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]), inputs[1])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input=arr target={inputs[1]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(jump_search)
print("jump_search defined.")

# Simplicity and clarity is Gold


<a id='6'></a>

## 6. 🧩 Pattern 3: Exponential Search — LC 702

---

```
PROBLEM:  Search in a sorted array of unknown/unbounded size.
          You can't set hi=n-1 if you don't know n. Exponential search solves this.

APPROACH: Double the search window (1→2→4→8→...) until arr[i] > target.
          Then binary search in [i//2, min(i, n-1)].

SLOW MOTION TRACE on arr=[1,2,4,8,16,32,64,128] target=32:
  i=1:  arr[1]=2  < 32  double
  i=2:  arr[2]=4  < 32  double
  i=4:  arr[4]=16 < 32  double
  i=8:  i >= len — clamp to len-1=7
  binary search in [4, 7]:
    mid=5  arr[5]=32=32 ✓  found at index 5

KEY INSIGHT: i doubles each step, so we do at most log(n) doublings.
             Binary search in the final window is also O(log n).
             Total: O(log n).

WHEN TO USE:
  - LC 702 (Search in sorted array of unknown size)
  - Infinite or streaming data where n is not known upfront

TIME / SPACE:
  Time:  O(log n) — log n doublings + log n binary search
  Space: O(1)
```


In [ ]:
from typing import List

def exponential_search(arr: List[int], target: int) -> int:
    """
    Exponential Search — O(log n) for sorted arrays, works on unbounded data.
    Approach: double window until overshoot, then binary search in that window.
    Args:
        arr (List[int]): sorted list.
        target (int): value to find.
    Returns:
        int: index of target, or -1 if not found.
    Time:  O(log n) — at most log n doublings + log n binary search
    Space: O(1)
    """
    n = len(arr)
    if n == 0:
        return -1
    if arr[0] == target:
        return 0               # check first element separately

    # Double i until arr[i] >= target or we go out of bounds
    i = 1
    while i < n and arr[i] < target:
        i *= 2                 # exponential window growth

    # Binary search in [i//2, min(i, n-1)]
    lo = i // 2
    hi = min(i, n - 1)
    return _binary_search(arr, target, lo, hi)

def _binary_search(arr, target, lo, hi):
    """Standard binary search in arr[lo..hi]."""
    while lo <= hi:
        mid = lo + (hi - lo) // 2    # avoids integer overflow (good habit)
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

# Slow motion on arr=[1,2,4,8,16,32,64,128] target=32:
# i=1: 2<32 double → i=2
# i=2: 4<32 double → i=4
# i=4: 16<32 double → i=8
# i=8: 8>=8=n, exit loop
# binary search [4, 7]: mid=5 arr[5]=32 ✓

def test_harness(fn):
    arr = [1, 2, 4, 8, 16, 32, 64, 128]
    tests = [
        (arr, 32, 5),
        (arr, 1, 0),
        (arr, 128, 7),
        (arr, 5, -1),
        ([], 1, -1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]), inputs[1])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | target={inputs[1]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(exponential_search)
print("exponential_search defined.")

# Simplicity and clarity is Gold


<a id='7'></a>

## 7. 🧩 Pattern 4: Binary Search — Standard Template — LC 704

---

```
PROBLEM:  Find target in sorted array. Return index or -1.

APPROACH: lo=0, hi=n-1. While lo<=hi: check mid.
          If match, return. If too small, lo=mid+1. If too big, hi=mid-1.

SLOW MOTION TRACE on arr=[1,3,5,7,9,11] target=7:
  lo=0  hi=5  mid=2  arr[2]=5 < 7  → lo=3
  lo=3  hi=5  mid=4  arr[4]=9 > 7  → hi=3
  lo=3  hi=3  mid=3  arr[3]=7 = 7  ✓ return 3

  target=6 (not in array):
  lo=0  hi=5  mid=2  arr[2]=5 < 6  → lo=3
  lo=3  hi=5  mid=4  arr[4]=9 > 6  → hi=3
  lo=3  hi=3  mid=3  arr[3]=7 > 6  → hi=2
  lo=3 > hi=2 → exit loop → return -1

THE MID FORMULA: mid = lo + (hi - lo) // 2
  NOT mid = (lo + hi) // 2  ← can integer overflow in languages with fixed int size

KEY INSIGHT: lo <= hi terminates when lo > hi (target absent).
             lo < hi would miss the last element case.

TIME / SPACE:
  Time:  O(log n) — halves the search space each iteration
  Space: O(1)     — iterative, no stack
```


In [ ]:
from typing import List

def binary_search(nums: List[int], target: int) -> int:
    """
    LC 704 — Binary Search (standard exact-match template).
    Approach: lo/hi pointers closing in; mid tested each iteration.
    Args:
        nums (List[int]): sorted list in ascending order.
        target (int): value to find.
    Returns:
        int: index of target if present, -1 otherwise.
    Time:  O(log n) — halves search space each step
    Space: O(1)     — iterative
    """
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2    # safe mid — no overflow risk
        if nums[mid] == target:
            return mid               # exact match
        elif nums[mid] < target:
            lo = mid + 1             # target is in right half
        else:
            hi = mid - 1             # target is in left half
    return -1                        # lo > hi — target not in array

# Slow motion on [-1,0,3,5,9,12] target=9:
# lo=0 hi=5 mid=2 nums[2]=3 < 9 → lo=3
# lo=3 hi=5 mid=4 nums[4]=9 = 9 ✓ return 4

def test_harness(fn):
    tests = [
        ([-1, 0, 3, 5, 9, 12], 9, 4),
        ([-1, 0, 3, 5, 9, 12], 2, -1),
        ([1], 1, 0),
        ([1], 2, -1),
        ([], 1, -1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]), inputs[1])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(binary_search)
print("binary_search defined.")

# Simplicity and clarity is Gold


<a id='8'></a>

## 8. 🧩 Pattern 5: Lower Bound (leftmost occurrence) — LC 34

---

```
PROBLEM:  Find the leftmost index where target appears (or where it would be inserted).
          Equivalent to Python's bisect_left.

APPROACH: When arr[mid] == target, do NOT return — keep going left (hi = mid).
          Loop ends when lo == hi, which is the leftmost valid position.

KEY: Use lo < hi (not lo <= hi) and hi = mid (not mid - 1).
     At the end, lo == hi == the answer.

SLOW MOTION TRACE on arr=[1,3,3,3,5,7] target=3:
  lo=0  hi=5  mid=2  arr[2]=3 == 3  → hi=2   (don't stop — look left)
  lo=0  hi=2  mid=1  arr[1]=3 == 3  → hi=1
  lo=0  hi=1  mid=0  arr[0]=1 < 3   → lo=1
  lo=1  hi=1  → exit  return lo=1  ✓ (leftmost 3 is at index 1)

  target=4 (not in array):
  lo=0  hi=5  mid=2  arr[2]=3 < 4   → lo=3
  lo=3  hi=5  mid=4  arr[4]=5 > 4   → hi=4
  lo=3  hi=4  mid=3  arr[3]=3 < 4   → lo=4
  lo=4  hi=4  → exit  return lo=4  (insertion point for 4)

KEY INSIGHT: lower_bound returns the first position where target could live.
             If arr[lo] != target, target is absent.

TIME / SPACE: O(log n) / O(1)
```


In [ ]:
from typing import List

def lower_bound(nums: List[int], target: int) -> int:
    """
    Lower Bound — leftmost index where target appears (or would be inserted).
    Equivalent to bisect_left(nums, target).
    Args:
        nums (List[int]): sorted list.
        target (int): value to find.
    Returns:
        int: leftmost index i such that nums[i] >= target. 0..len(nums).
    Time:  O(log n)
    Space: O(1)
    """
    lo, hi = 0, len(nums)          # hi = len, not len-1 — insertion past end is valid
    while lo < hi:                 # lo < hi, NOT lo <= hi
        mid = lo + (hi - lo) // 2
        if nums[mid] < target:     # strict <: mid is too small, must go right
            lo = mid + 1
        else:                      # nums[mid] >= target: mid could be the answer, keep it
            hi = mid               # hi = mid, NOT mid-1
    return lo                      # lo == hi == leftmost position

# Slow motion on [1,3,3,3,5,7] target=3:
# lo=0 hi=6 mid=3 nums[3]=3>=3 → hi=3
# lo=0 hi=3 mid=1 nums[1]=3>=3 → hi=1
# lo=0 hi=1 mid=0 nums[0]=1<3  → lo=1
# lo=1 hi=1 exit → return 1 ✓

def upper_bound(nums: List[int], target: int) -> int:
    """
    Upper Bound — first index after rightmost target.
    Equivalent to bisect_right(nums, target).
    Returns: first index i such that nums[i] > target.
    subtract 1 to get the rightmost target index.
    """
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if nums[mid] <= target:    # <= : mid is still <= target, go right
            lo = mid + 1
        else:
            hi = mid
    return lo                      # first position strictly after target

def find_range(nums: List[int], target: int):
    """Return [first_index, last_index] of target, or [-1, -1] if absent."""
    left  = lower_bound(nums, target)
    if left == len(nums) or nums[left] != target:
        return [-1, -1]            # target not found
    right = upper_bound(nums, target) - 1   # last index of target
    return [left, right]

def test_harness_bounds(fn_lb, fn_ub):
    arr = [1, 3, 3, 3, 5, 7]
    print(f"lower_bound(arr, 3) = {fn_lb(arr, 3)}")    # 1
    print(f"lower_bound(arr, 4) = {fn_lb(arr, 4)}")    # 4 (insertion point)
    print(f"upper_bound(arr, 3) = {fn_ub(arr, 3)}")    # 4 (first after 3s)
    print(f"upper_bound(arr, 7) = {fn_ub(arr, 7)}")    # 6 (after last element)

test_harness_bounds(lower_bound, upper_bound)

def test_harness_range(fn):
    tests = [
        ([5, 7, 7, 8, 8, 10], 8, [3, 4]),
        ([5, 7, 7, 8, 8, 10], 6, [-1, -1]),
        ([], 0, [-1, -1]),
        ([1], 1, [0, 0]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_range(find_range)
print("lower_bound, upper_bound, find_range defined.")

# Simplicity and clarity is Gold


<a id='10'></a>

## 10. 🧩 Pattern 7: Search on Answer Space — LC 875, 410, 1011

---

```
PROBLEM:  "Find the minimum X such that condition(X) is true."
          The answer isn't in an array — it's a number in a range [lo, hi].

APPROACH: Binary search on the ANSWER, not on an array index.
          Define a feasibility function: can_do(x) → True/False.
          Binary search for the leftmost x where can_do(x) is True.

TEMPLATE:
  lo = minimum possible answer
  hi = maximum possible answer
  while lo < hi:
      mid = (lo + hi) // 2
      if feasible(mid):
          hi = mid         # mid works, but maybe smaller works too
      else:
          lo = mid + 1     # mid doesn't work, need bigger
  return lo                # smallest feasible value

LC 875 — Koko Eating Bananas:
  "minimum speed k such that Koko finishes in h hours"
  feasible(k) = sum(ceil(pile/k) for pile in piles) <= h
  lo=1, hi=max(piles)
  binary search → O(n log max_pile)

LC 410 — Split Array Largest Sum:
  "minimum largest sum when splitting array into m subarrays"
  feasible(max_sum) = can we split so no subarray exceeds max_sum?
  lo=max(nums), hi=sum(nums)

KEY INSIGHT: Any "minimize the maximum" or "maximize the minimum" problem
             hints at binary search on answer space.

TIME / SPACE:
  Time:  O(n log(hi-lo)) — log(answer_range) binary steps, O(n) per feasibility check
  Space: O(1)
```


In [ ]:
from typing import List
import math

def minEatingSpeed(piles: List[int], h: int) -> int:
    """
    LC 875 — Koko Eating Bananas.
    Approach: binary search on speed k in [1, max(piles)].
    Feasibility: total hours at speed k <= h.
    Args:
        piles (List[int]): banana pile sizes.
        h (int): total hours available.
    Returns:
        int: minimum eating speed k (bananas per hour).
    Time:  O(n log m) — n=piles, m=max(piles); log m binary steps, O(n) per step
    Space: O(1)
    """
    def feasible(speed: int) -> bool:
        # hours needed at this speed = sum of ceil(pile / speed)
        return sum(math.ceil(pile / speed) for pile in piles) <= h

    lo, hi = 1, max(piles)            # answer range: 1 banana/hr to max pile
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if feasible(mid):              # mid works — but can we go slower?
            hi = mid                   # keep mid as candidate, search left
        else:
            lo = mid + 1               # mid too slow, need faster speed
    return lo                          # smallest feasible speed

# Slow motion on piles=[3,6,7,11] h=8:
# lo=1 hi=11 mid=6: ceil(3/6)+ceil(6/6)+ceil(7/6)+ceil(11/6)=1+1+2+2=6<=8 feasible→hi=6
# lo=1 hi=6  mid=3: ceil(3/3)+ceil(6/3)+ceil(7/3)+ceil(11/3)=1+2+3+4=10>8 not feasible→lo=4
# lo=4 hi=6  mid=5: 1+2+2+3=8<=8 feasible→hi=5
# lo=4 hi=5  mid=4: 1+2+2+3=8<=8 feasible→hi=4
# lo=4 hi=4 → return 4 ✓

def test_harness_koko(fn):
    tests = [
        ([3, 6, 7, 11], 8, 4),
        ([30, 11, 23, 4, 20], 5, 30),
        ([30, 11, 23, 4, 20], 6, 23),
        ([1000000000], 2, 500000000),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_koko(minEatingSpeed)
print("minEatingSpeed defined.")

# ── Answer space: split array largest sum (LC 410) ────────────────────────────
def splitArray(nums: List[int], k: int) -> int:
    """
    LC 410 — Split Array Largest Sum.
    Approach: binary search on max_sum; feasibility = can split into <= k pieces.
    Args:
        nums (List[int]): array of positive integers.
        k (int): number of subarrays.
    Returns:
        int: minimized largest subarray sum.
    Time:  O(n log(sum)) — log(sum-max) binary steps, O(n) per feasibility check
    Space: O(1)
    """
    def feasible(max_sum: int) -> bool:
        pieces, current = 1, 0
        for x in nums:
            if current + x > max_sum:  # adding x would exceed limit — start new piece
                pieces += 1
                current = x
                if pieces > k:         # too many pieces — max_sum too small
                    return False
            else:
                current += x
        return True

    lo, hi = max(nums), sum(nums)      # minimum = max single element, maximum = whole array
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

def test_harness_split(fn):
    tests = [
        ([7, 2, 5, 10, 8], 2, 18),
        ([1, 2, 3, 4, 5], 2, 9),
        ([1, 4, 4], 3, 4),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_split(splitArray)
print("splitArray defined.")

# Simplicity and clarity is Gold


<a id='11'></a>

## 11. 🧩 Pattern 8: Rotated Sorted Array — LC 33, 153

---

```
PROBLEM:  Array was sorted then rotated at unknown pivot. Search for target.
          [4,5,6,7,0,1,2] — sorted [0..7] rotated at pivot=4.

APPROACH: One-pass binary search. At each mid, determine WHICH HALF is sorted.
          The key insight: one of [lo..mid] or [mid..hi] is always sorted.

CASE ANALYSIS at mid:
  If arr[mid] >= arr[lo]:            ← left half [lo..mid] is sorted
      if arr[lo] <= target < arr[mid]: search left (hi = mid - 1)
      else: search right (lo = mid + 1)
  Else:                              ← right half [mid..hi] is sorted
      if arr[mid] < target <= arr[hi]: search right (lo = mid + 1)
      else: search left (hi = mid - 1)

SLOW MOTION TRACE on arr=[4,5,6,7,0,1,2] target=0:
  lo=0 hi=6 mid=3 arr[3]=7 >= arr[0]=4  → left sorted [4..7]
    target=0 not in [4,7) → lo=4
  lo=4 hi=6 mid=5 arr[5]=1 >= arr[4]=0  → left sorted [0,1]
    target=0 == arr[lo]=0 at boundary → arr[lo]==0 found at lo=4 ✓

Find minimum in rotated (LC 153):
  Binary search for the pivot — the minimum is always in the unsorted half.
  if arr[mid] > arr[hi]: min is in right half (lo = mid+1)
  else: min could be mid or left (hi = mid)

TIME / SPACE: O(log n) / O(1)
```


In [ ]:
from typing import List

def search_rotated(nums: List[int], target: int) -> int:
    """
    LC 33 — Search in Rotated Sorted Array.
    Approach: binary search with half-sorted analysis.
    Args:
        nums (List[int]): rotated sorted array (distinct values).
        target (int): value to find.
    Returns:
        int: index of target, or -1 if absent.
    Time:  O(log n)
    Space: O(1)
    """
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if nums[mid] == target:
            return mid

        if nums[lo] <= nums[mid]:          # left half [lo..mid] is sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1              # target in sorted left half
            else:
                lo = mid + 1              # target in right half
        else:                              # right half [mid..hi] is sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1              # target in sorted right half
            else:
                hi = mid - 1              # target in left half
    return -1

def find_min_rotated(nums: List[int]) -> int:
    """
    LC 153 — Find Minimum in Rotated Sorted Array.
    Approach: binary search for pivot; minimum is always in unsorted half.
    Args:
        nums (List[int]): rotated sorted array.
    Returns:
        int: minimum value.
    Time:  O(log n)
    Space: O(1)
    """
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if nums[mid] > nums[hi]:   # mid is in left (larger) portion — min is right
            lo = mid + 1
        else:                      # mid is in right (smaller) portion — min could be mid
            hi = mid
    return nums[lo]

# Slow motion LC 33 on [4,5,6,7,0,1,2] target=0:
# lo=0 hi=6 mid=3 nums[3]=7≠0  nums[0]=4<=nums[3]=7 → left sorted
#   target=0 not in [4,7) → lo=4
# lo=4 hi=6 mid=5 nums[5]=1≠0  nums[4]=0<=nums[5]=1 → left sorted
#   target=0 in [0,1) → hi=4
# lo=4 hi=4 mid=4 nums[4]=0=0 ✓ return 4

def test_harness_search(fn):
    tests = [
        ([4,5,6,7,0,1,2], 0, 4),
        ([4,5,6,7,0,1,2], 3, -1),
        ([1], 0, -1),
        ([1], 1, 0),
        ([3,1], 1, 1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed (search_rotated)")

def test_harness_min(fn):
    tests = [
        ([3,4,5,1,2], 1),
        ([4,5,6,7,0,1,2], 0),
        ([11,13,15,17], 11),   # not rotated
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed (find_min_rotated)")

test_harness_search(search_rotated)
test_harness_min(find_min_rotated)
print("search_rotated, find_min_rotated defined.")

# Simplicity and clarity is Gold


<a id='12'></a>

## 12. 🧩 Pattern 9: bisect module — Python stdlib

---

```
PROBLEM:  Use Python's built-in binary search tools instead of re-implementing.

bisect_left(a, x)   → leftmost index where x can be inserted (= lower_bound)
bisect_right(a, x)  → rightmost index where x can be inserted (= upper_bound)
insort_left(a, x)   → insert x maintaining sorted order (O(n) due to list shift)
insort_right(a, x)  → same, but rightmost position

SLOW MOTION on arr=[1,3,3,3,5,7] x=3:
  bisect_left  → 1   (leftmost position to insert 3)
  bisect_right → 4   (rightmost position to insert 3)

  arr[bisect_left(arr,3)]  = 3   (first 3)
  arr[bisect_right(arr,3)-1] = 3 (last 3)

CHECKING PRESENCE:
  idx = bisect_left(arr, x)
  found = idx < len(arr) and arr[idx] == x

KEY INSIGHT: bisect_left / bisect_right replace both lower_bound and upper_bound.
             Use them in interviews — interviewers expect you to know the stdlib.
```


In [ ]:
from bisect import bisect_left, bisect_right, insort_left, insort_right
from typing import List

arr = [1, 3, 3, 3, 5, 7]
print(f"arr = {arr}")

# bisect_left — where would 3 go to keep sorted? (leftmost)
print(f"bisect_left(arr, 3)  = {bisect_left(arr, 3)}")    # 1
print(f"bisect_left(arr, 4)  = {bisect_left(arr, 4)}")    # 4 (insertion point)
print(f"bisect_left(arr, 0)  = {bisect_left(arr, 0)}")    # 0 (before everything)
print(f"bisect_left(arr, 99) = {bisect_left(arr, 99)}")   # 6 (after everything)

print()

# bisect_right — rightmost insertion point
print(f"bisect_right(arr, 3) = {bisect_right(arr, 3)}")   # 4
print(f"bisect_right(arr, 4) = {bisect_right(arr, 4)}")   # 4 (same as left for missing)

print()

# Presence check using bisect_left
def contains(arr, x):
    idx = bisect_left(arr, x)
    return idx < len(arr) and arr[idx] == x

print(f"contains(arr, 3) = {contains(arr, 3)}")    # True
print(f"contains(arr, 4) = {contains(arr, 4)}")    # False

# First and last occurrence
def find_first(arr, x):
    idx = bisect_left(arr, x)
    return idx if idx < len(arr) and arr[idx] == x else -1

def find_last(arr, x):
    idx = bisect_right(arr, x) - 1
    return idx if idx >= 0 and arr[idx] == x else -1

print(f"find_first(arr, 3) = {find_first(arr, 3)}")   # 1
print(f"find_last(arr,  3) = {find_last(arr, 3)}")    # 3

print()

# insort — maintain sorted order during insertions
sorted_stream = []
for val in [5, 2, 8, 1, 9, 3]:
    insort_left(sorted_stream, val)    # O(n) insert but O(log n) search position
    print(f"  insert {val} → {sorted_stream}")

# LC 35 — Search Insert Position = bisect_left
def searchInsert(nums: List[int], target: int) -> int:
    """LC 35 — one line with bisect_left."""
    return bisect_left(nums, target)

print()
print(f"searchInsert([1,3,5,6], 5) = {searchInsert([1,3,5,6], 5)}")   # 2
print(f"searchInsert([1,3,5,6], 2) = {searchInsert([1,3,5,6], 2)}")   # 1
print(f"searchInsert([1,3,5,6], 7) = {searchInsert([1,3,5,6], 7)}")   # 4

# Simplicity and clarity is Gold


<a id='15'></a>

## 15. Full Decision Map

```
QUESTION TYPE                              KEY TECHNIQUE              LC
──────────────────────────────────────────────────────────────────────────────
Find exact value in sorted array           Binary Search              704
Leftmost occurrence / insert position      bisect_left / lower_bound  34, 35
Rightmost occurrence                       bisect_right - 1           34
Search rotated sorted array                Binary Search + half check 33
Find minimum in rotated array              Binary Search (pivot side) 153
Find peak element                          Binary Search (slope side) 162
Minimize speed/capacity/answer             Answer Space Binary Search 875, 1011
Minimize max / maximize min                Answer Space Binary Search 410, 2064
Unknown array size / unbounded             Exponential Search         702
Unsorted, any type, small n                Linear Search              —
Sorted, cache-friendly, no division        Jump Search                —
Python sorted array: insert/lookup         bisect module              35
──────────────────────────────────────────────────────────────────────────────
```


<a id='16'></a>

## 16. Interview Cheat Sheet

### 1. When to reach for binary search:

| Signal | What to Do |
|--------|------------|
| Sorted array + exact match | Standard binary search |
| "first/last occurrence" | lower_bound / upper_bound |
| "minimum X where condition" | Answer space binary search |
| Rotated sorted array | One-pass binary search with half check |
| Unknown size array | Exponential search |
| Python, just need position | bisect_left / bisect_right |

### 2. The four binary search templates:

```python
# EXACT MATCH
lo, hi = 0, len(nums) - 1
while lo <= hi:
    mid = lo + (hi - lo) // 2
    if nums[mid] == target: return mid
    elif nums[mid] < target: lo = mid + 1
    else: hi = mid - 1
return -1

# LOWER BOUND (leftmost / bisect_left)
lo, hi = 0, len(nums)
while lo < hi:
    mid = lo + (hi - lo) // 2
    if nums[mid] < target: lo = mid + 1
    else: hi = mid
return lo

# UPPER BOUND (bisect_right)
lo, hi = 0, len(nums)
while lo < hi:
    mid = lo + (hi - lo) // 2
    if nums[mid] <= target: lo = mid + 1
    else: hi = mid
return lo

# ANSWER SPACE (minimize feasible)
lo, hi = min_possible, max_possible
while lo < hi:
    mid = lo + (hi - lo) // 2
    if feasible(mid): hi = mid
    else: lo = mid + 1
return lo
```

### 3. Gotchas to not forget:

```
❌  mid = (lo + hi) // 2  can overflow in C/Java (use lo + (hi-lo)//2)
❌  lo <= hi for exact match; lo < hi for lower/upper bound — mixing them causes bugs
❌  Binary search requires SORTED input — always confirm this
❌  Answer space lo/hi must BRACKET the answer — set them carefully
✅  bisect_left = lower_bound; bisect_right = upper_bound
✅  After lower_bound: check arr[lo] == target before claiming found
✅  For "minimize the maximum" → answer space with feasible() check
✅  Rotated array: one half is always sorted — use that fact
```


```
SEARCH ALGORITHMS MASTER MAP
══════════════════════════════════════════════════════

                     SEARCH
                       │
         ┌─────────────┼──────────────┐
         ▼             ▼              ▼
    UNSORTED      SORTED ARRAY    ANSWER SPACE
    ARRAY              │           (not array)
         │        ┌────┴────┐           │
    Linear        │         │     Binary search
    O(n)     Standard    Variants   on value range
              Binary      │          lo=min
              O(log n)  ┌─┴───────┐  hi=max
                        │         │  feasible(mid)?
                   Lower/Upper  Rotated    │
                   Bound        Array   hi=mid or lo=mid+1
                   bisect_left  LC 33
                   bisect_right LC 153
                        │
                   Jump Search  O(√n)
                   Exponential  O(log n) unbounded
```

---
*End of Search Algorithms Master Guide — Sean Edition*
